# Your First Vector RAG Application: Cat Health Assistant

In this notebook, we will build a dense vector retrieval application using **LangChain v1**, **OpenAI embeddings**, and **Qdrant** as an in-memory vector database.

The goal is to understand the core RAG loop:

1. Load a cat health guideline PDF
2. Split it into smaller chunks
3. Embed those chunks
4. Store the embeddings in Qdrant
5. Retrieve relevant chunks for a question
6. Generate an answer grounded in the retrieved context

> Note: This notebook expects Python 3.12 and uses uv for dependency management.

> Note: This is a vector RAG lesson, not a veterinary care tool. The assistant should answer from the PDF and point users to a veterinarian for diagnosis, treatment, medication, or urgent care decisions.

## Table of Contents

- Task 1: Environment Setup
- Task 2: Embedding Similarity Primer
- Task 3: Documents - Loading the Cat Health Guideline PDF
- Task 4: Chunking the Documents
- Task 5: Embeddings and Qdrant
- Task 6: Retrieval with Scores
- Task 7: Retrieval Augmented Generation
- Activity: Tune Retrieval Quality

## Task 1: Environment Setup

From the `01_Dense_Vector_Retrieval` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

### Imports

LangChain v1 separates integrations into partner packages. We will use:

- `langchain_community` for PDF loading
- `langchain_text_splitters` for chunking
- `langchain_openai` for chat and embedding models
- `langchain_qdrant` for the Qdrant vector store

In [1]:
from pathlib import Path
from math import sqrt
from getpass import getpass
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/z8/pbvjy1gs1d3b_rd53b4knls80000gp/T/ipykernel_13240/4147450701.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### OpenAI API Key

The chat model and embedding model both use OpenAI. If `OPENAI_API_KEY` is not already set in your environment, this cell will ask for it securely.

In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Task 2: Embedding Similarity Primer

Before we load a full PDF, let's make dense vector retrieval less mysterious.

An embedding model turns text into a list of numbers. Texts with related meaning should land closer together in that vector space.

A common way to score closeness is **cosine similarity**:

```text
cosine_similarity(a, b) = dot_product(a, b) / (length(a) * length(b))
```

The intuition: if two vectors point in a similar direction, their cosine similarity is higher. Vector databases like Qdrant use this same idea, but at a much larger scale.

In [3]:
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=embedding_model)

example_texts = [
    "king",
    "queen",
    "banana",
    "cat",
    "veterinarian",
    "cat health guidelines",
]

example_vectors = dict(zip(example_texts, embeddings.embed_documents(example_texts)))


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    length_a = sqrt(sum(a * a for a in vector_a))
    length_b = sqrt(sum(b * b for b in vector_b))
    return dot_product / (length_a * length_b)


comparison_pairs = [
    ("king", "queen"),
    ("king", "banana"),
    ("cat", "veterinarian"),
    ("cat", "cat health guidelines"),
]

for left, right in comparison_pairs:
    score = cosine_similarity(example_vectors[left], example_vectors[right])
    print(f"{left:>22} <> {right:<22} score={score:.3f}")

                  king <> queen                  score=0.591
                  king <> banana                 score=0.310
                   cat <> veterinarian           score=0.356
                   cat <> cat health guidelines  score=0.496


A few important notes:

- The score is useful for ranking, not as an absolute truth about meaning.
- Different embedding models can produce different scores.
- In RAG, we embed each document chunk once, then embed the user's query and search for the nearest chunk vectors.

That is the retrieval part of RAG.

## Task 3: Documents

LangChain represents loaded text as `Document` objects. A `Document` has:

- `page_content`: the text
- `metadata`: information such as source file and page number

We will load one `Document` per PDF page, then split those pages into smaller chunks.

### Course PDF

This notebook uses the bundled cat health guideline PDF at:

```text
01_Dense_Vector_Retrieval/data/cat_health_guidelines.pdf
```

The next cell checks that the course material is present before we start loading pages.

In [4]:
pdf_path = Path("data/cat_health_guidelines.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(
        f"Expected the cat health guideline PDF at: {pdf_path.resolve()}\n"
        "The bundled course PDF is missing from this copy of the materials."
    )

### Load the PDF

`PyPDFLoader` extracts text from text-based PDFs. If the PDF is scanned images, this loader may return little or no text, and OCR would be needed.

In [5]:
loader = PyPDFLoader(str(pdf_path))
pages = loader.load()

for page in pages:
    page.metadata["source"] = pdf_path.name
    page.metadata["document_type"] = "cat_health_guideline"

pages = [page for page in pages if page.page_content.strip()]

if not pages:
    raise ValueError(
        "The PDF loaded, but no extractable text was found. "
        "This usually means the PDF is scanned and needs OCR first."
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")

Loaded 22 text-containing PDF pages.


In [6]:
print(pages[0].page_content[:750])
print("\nMetadata:", pages[0].metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #1

Why is metadata important for a RAG application?

##### ✅ Answer:

Metadata is what turns a retrieved chunk from "a blob of text" into a **traceable, filterable, and trustworthy** piece of evidence. It matters for several concrete reasons:

- **Citations and trust.** Fields like `source`, `page`, and `start_index` let us tell the user *where* an answer came from (e.g., "page 8 of `cat_health_guidelines.pdf`"). Without this, users have no way to verify the LLM's output and we cannot detect hallucinations.
- **Filtering and routing.** Metadata such as `document_type: "cat_health_guideline"` lets the vector store pre-filter the search space (e.g., only feline documents, only recent versions, only a given customer's documents). This improves both relevance and security/multi-tenancy.
- **Debugging retrieval.** When an answer looks wrong, metadata tells us which page/section was retrieved so we can inspect chunking, overlap, or the source PDF itself.
- **Re-ranking and weighting.** Metadata (recency, author, document quality) can be used as features in a re-ranker on top of vector similarity.
- **Lifecycle management.** We need metadata to update or delete documents in the index when the source PDF changes — otherwise the store silently drifts out of date.
- **Domain safety.** For a health/medical assistant, knowing the source document and page is essential so the assistant (and the user) can defer to the authoritative guideline and to a veterinarian.

## Task 4: Chunking the Documents

A full PDF page can be too large or too mixed-topic for high-quality retrieval. We split pages into overlapping chunks so each chunk has enough local context but is still focused.

Here we will start with chunks of 1,000 characters and 200 characters of overlap. The chunk size controls how much text each vector represents; the overlap keeps nearby context from being lost at chunk boundaries.

`RecursiveCharacterTextSplitter` tries to split on natural boundaries first, such as paragraphs and line breaks, before falling back to smaller separators.

In [7]:
chunk_size = 1000
chunk_overlap = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
)

splits = text_splitter.split_documents(pages)

print(f"Split {len(pages)} pages into {len(splits)} chunks.")
print(f"Chunk size: {chunk_size} characters")
print(f"Chunk overlap: {chunk_overlap} characters")

Split 22 pages into 135 chunks.
Chunk size: 1000 characters
Chunk overlap: 200 characters


In [8]:
sample_chunk = splits[0]
print(sample_chunk.page_content[:750])
print("\nMetadata:", sample_chunk.metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #2

What tradeoff do we make when choosing chunk size and chunk overlap?

##### ✅ Answer:

Chunking is a **context vs. precision** tradeoff, and overlap is a **continuity vs. cost** tradeoff.

**Chunk size:**

- **Larger chunks** preserve more surrounding context inside a single vector, so the LLM gets richer, self-contained passages and is less likely to be missing a definition or a qualifying sentence. The downside is that each vector represents a *mixture* of topics, which dilutes the embedding — the chunk can match many queries weakly instead of one query strongly, hurting ranking precision. Larger chunks also burn more prompt tokens per retrieved result.
- **Smaller chunks** produce sharper, more topically focused embeddings and better top-k precision, but each chunk may be too fragmentary to actually answer the question on its own (e.g., a sentence that says "this is contraindicated" without saying *what* "this" is). You then typically need a larger `k` to reassemble the context, which adds noise and cost.

**Chunk overlap:**

- **More overlap** reduces the risk that an important sentence gets split exactly at a chunk boundary and loses its surrounding context. It improves recall on boundary-spanning facts.
- **More overlap** also means more chunks → more embeddings to compute and store, more duplicated text in retrieved results, and a higher chance the top-k is filled with near-duplicates of the same passage.

**Rule of thumb for this notebook:** 1,000 chars with 200-char overlap is a reasonable starting point for prose PDFs. For dense reference material (tables, definitions, dosing) smaller chunks (~300–500 chars) with proportionally smaller overlap usually rank better; for narrative or explanatory text, larger chunks tend to give the LLM enough context to answer in one shot.

Use the [Chunk Visualizer](https://chunkviz.up.railway.app/) to experiment with different chunk sizes and overlaps and see how the text boundaries change.

## Task 5: Embeddings and Qdrant

Now we apply the same embedding idea to every chunk from the PDF. Qdrant stores those vectors and lets us search for chunks that are close to a query in embedding space.

We already created an OpenAI embedding model in the primer above. The Qdrant collection name is just a label for the set of vectors we are creating.

For this notebook, Qdrant runs in memory with `location=":memory:"`. That means no Docker, no Qdrant Cloud account, and no persistence after the notebook kernel stops.

In [9]:
collection_name = "cat_health_guidelines"

vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
    force_recreate=True,
)

print(f"Embedded chunks with: {embedding_model}")
print(f"Built in-memory Qdrant collection: {collection_name}")

Embedded chunks with: text-embedding-3-small
Built in-memory Qdrant collection: cat_health_guidelines


## Task 6: Retrieval with Scores

Before we generate answers, we should inspect retrieval directly. If retrieval returns poor context, the final answer will usually be poor too.

The value `k` controls how many chunks the retriever returns. A larger `k` gives the model more context, but it can also add noise. We will start with `k = 4` and tune it later.

Qdrant can return both the matching `Document` and a similarity score. This is the same ranking idea we saw with `king`, `queen`, and `cat`, now applied to PDF chunks.

In [10]:
def display_retrieval_results(query: str, k: int) -> list[tuple]:
    """Retrieve chunks and print a compact view of the results."""
    results = vector_store.similarity_search_with_score(query, k=k)

    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("\n", " ")

        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)

    return results

In [11]:
retrieval_k = 4
retrieval_query = "What signs suggest that a cat should be seen by a veterinarian?"
retrieved_results = display_retrieval_results(retrieval_query, k=retrieval_k)

Source 1 | score=0.584 | page=8 | start_index=0
Detecting signs of pain or anxiety and evaluation of quality of life are most commonly of concern in the mature adult or senior cat but may be relevant at any life stage. During the physical examination, particular focus is on pain assessment and abdominal and thyroid palpation. A detailed mus- culoskeletal examination to detect signs of osteoarthr
--------------------------------------------------------------------------------
Source 2 | score=0.571 | page=7 | start_index=2384
Asking speci ﬁc questions concerning whether vomiting, vom- iting hairballs, or diarrhea is occurring, and the frequency of each, is recommended as some clients may consider vomiting or vomiting hairballs to be normal for their cat. Additionally, discuss the im- portance of monitoring weight, and ask about any chronic enter- opathy or gastrointesti
--------------------------------------------------------------------------------
Source 3 | score=0.565 | page=7 | sta

#### ❓Question #3

What does a similarity score help you understand, and what does it not prove by itself?

##### ✅ Answer:

**What it helps with:**

- **Relative ranking.** It tells us which chunk is *closer* to the query than another, in the geometry of this specific embedding model. That ordering is the whole basis of vector retrieval.
- **A sanity signal for tuning.** Watching scores rise or fall as we change the query wording, chunk size, or embedding model gives quick feedback on whether retrieval is getting better.
- **A rough confidence floor.** If the top score is very low (or the gap between top-1 and top-k is tiny), retrieval is probably weak and the answer is more likely to be ungrounded.

**What it does *not* prove:**

- **It is not absolute truth or factual correctness.** A high cosine score only means the two vectors point in similar directions in *this* embedding space — it does not mean the chunk actually answers the question, nor that the chunk's content is correct.
- **It is not comparable across models.** A 0.58 from `text-embedding-3-small` is not the same as a 0.58 from another model. Thresholds must be re-calibrated per model.
- **It is not calibrated probability.** Cosine similarity is not a probability of relevance. There is no universal "good > 0.8" rule, especially with OpenAI embeddings, which often produce scores in a narrower band.
- **It does not distinguish topical match from answer match.** A chunk can be highly similar because it shares vocabulary with the query (e.g., "cat" and "veterinarian") while not containing the actual answer. That is why we still inspect content, use re-ranking, or evaluate with task-level metrics.

## Task 7: Retrieval Augmented Generation

Now we combine retrieval with generation. We will use a two-step RAG pattern:

1. Retrieve relevant chunks from Qdrant
2. Put those chunks into the prompt and ask the model to answer from the context

This is intentionally simpler than an agent. We always retrieve before answering, which makes the vector retrieval mechanics easy to inspect.

For generation, we will use `gpt-5.4-mini`.

In [12]:
chat_model = "gpt-5.4-mini"
llm = ChatOpenAI(model=chat_model)

RAG_SYSTEM_PROMPT = """You are a cat health guideline assistant in a vector RAG lesson.

Use only the provided context to answer the user's question.
If the context does not contain enough information, say: "I don't have enough information in the provided cat health guideline PDF to answer that."

Cite the retrieved sources inline using labels like [Source 1] or [Source 2].
Do not diagnose, prescribe medication, or replace a veterinarian.
For diagnosis, treatment decisions, medication questions, or urgent symptoms, recommend contacting a veterinarian.
Keep the answer concise and practical."""

RAG_USER_PROMPT = """Context:
{context}

Question: {question}

Answer from the context above."""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", RAG_USER_PROMPT),
    ]
)

rag_chain = rag_prompt | llm | StrOutputParser()

In [13]:
def format_context(scored_docs: list[tuple]) -> str:
    """Convert retrieved documents into a source-labeled context string."""
    formatted_chunks = []

    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        source = doc.metadata.get("source", "unknown source")

        formatted_chunks.append(
            f"[Source {index}] {source}, page {page_display}, score {score:.3f}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)


def answer_question(question: str, k: int) -> dict:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    scored_docs = vector_store.similarity_search_with_score(question, k=k)
    context = format_context(scored_docs)
    answer = rag_chain.invoke({"context": context, "question": question})

    sources = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": doc.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "start_index": doc.metadata.get("start_index"),
                "score": score,
            }
        )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context": scored_docs,
    }

Before calling the model, inspect the formatted context. This is the exact text that will be inserted into the RAG prompt.

In [14]:
example_context = format_context(retrieved_results[:2])
print(example_context[:2000])

[Source 1] cat_health_guidelines.pdf, page 8, score 0.584
Detecting signs of pain or anxiety and evaluation of quality of life
are most commonly of concern in the mature adult or senior cat but
may be relevant at any life stage.
During the physical examination, particular focus is on pain
assessment and abdominal and thyroid palpation. A detailed mus-
culoskeletal examination to detect signs of osteoarthritis is critical as
this condition is one of the most signi ﬁcant and underdiagnosed
diseases in cats.
23,28 A fundic examination is key to detecting signs of
ophthalmic disease or hypertension. 29 Practices should employ a
validated pain assessment scale or tool to diagnose, monitor, and
assist in the evaluation of patients for subtle signs of pain.
30
Changes in grooming habits, particularly increased grooming,
may signal a dermatologic issue such as atopy, food allergy, an
immune-mediated skin condition, infectious or parasitic disease,
endocrine condition, or paraneoplastic syndrom

In [15]:
answer_k = 4

result = answer_question(
    "What are signs that my cat may need veterinary attention?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)

Signs in the provided guideline that may warrant veterinary attention include:

- **Pain or anxiety/stress signs** such as cowering, crouching, crawling, muscular tension, hiding, frantic fleeing, flat or rotated-back ears, dilated pupils, forward whiskers, hissing/yowling/growling/screaming, or a rapid respiratory rate [Source 4]
- **Changes in grooming**: increased grooming or reduced grooming [Sources 1, 2]
- **Reduced grooming with possible illness signs**, including bladder pain, degenerative joint disease (DJD) pain, or reduced mobility [Source 2]
- **Changes in appetite, drinking, or urination**, including polyuria and polydipsia [Source 3]
- **Vomiting, vomiting hairballs, or diarrhea**, especially if frequent or new [Source 3]
- **Behavior changes**, such as increased nocturnal activity, vocalization, or changes in normal habits/activity [Source 3]
- **House-soiling or aggression** toward people or other animals [Source 2]

If you’re seeing any of these, especially if they are

### Vibe Check Queries

Run a few questions that should be answerable from a cat health guideline PDF. Then run one question that may not be answerable and confirm the assistant says it does not have enough information.

In [16]:
vibe_check_questions = [
    "What preventive care is recommended for cats?",
    "What symptoms should make me call a veterinarian?",
    "What should I know about feeding a healthy adult cat?",
    "Can my cat help me file my taxes?",
]

for question in vibe_check_questions:
    print("Question:", question)
    print(answer_question(question, k=answer_k)["answer"])
    print("=" * 100)

Question: What preventive care is recommended for cats?
The guidelines recommend at least annual veterinary examinations for all cats, with more frequent visits based on individual needs; senior cats should be seen at least every 6 months [Source 4]. Preventive care also includes individualized risk assessment and preventive healthcare strategies over the cat’s lifetime [Source 2]. For parasite prevention, routine broad-spectrum products are likely beneficial for most pet cats, and cats with outdoor exposure, travel, or boarding may need additional prevention measures; housemates may need synchronized treatment if one cat is newly acquired [Source 1].
Question: What symptoms should make me call a veterinarian?
You should call a veterinarian if your cat has changes in appetite, polyuria/polydipsia, vomiting, vomiting hairballs, diarrhea, increased nocturnal activity or vocalization, or changes in normal habits or activity. These signs may indicate disease, pain, cognitive dysfunction, r

#### ❓Question #4

For the vibe check queries above, did the retrieved context seem relevant before generation? Why or why not?

##### ✅ Answer:

Mostly yes, with one expected miss:

- **"What preventive care is recommended for cats?"** — Retrieved chunks covered annual/biannual exams, individualized risk assessment, and parasite prevention. Highly relevant; the question maps cleanly onto vocabulary that appears throughout the AAHA/AAFP guideline.
- **"What symptoms should make me call a veterinarian?"** — Retrieved chunks on vomiting, diarrhea, appetite changes, polyuria/polydipsia, grooming changes, and behavior changes were on-topic. Relevant. Some of the chunks were oriented toward *clinicians* asking the *owner* questions rather than owner-facing advice, but the underlying signs are the same.
- **"What should I know about feeding a healthy adult cat?"** — Retrieved chunks on body condition score, RER/DER calculations, AAFCO labeling, and gradual diet transitions are directly relevant nutrition content.
- **"Can my cat help me file my taxes?"** — Retrieval still returned the four nearest chunks (the vector store always returns `k` results), and their similarity scores were low and the content was unrelated. This is the expected behavior: **vector search is always best-effort and never returns "nothing"**, so it is the *prompt* and the LLM that correctly refused to answer. This is exactly why we keep the "I don't have enough information…" instruction in the system prompt and inspect scores, not just rely on retrieval to filter junk queries.

**Why relevance was generally good:** the queries use plain-English health/care vocabulary that overlaps strongly with the guideline's terminology, the chunks are large enough (1,000 chars) to carry self-contained context, and `k=4` is enough variety to cover multi-faceted questions like "what symptoms" without drowning the prompt in noise.

**Where it could fail:** queries that use synonyms the document does not use, queries that require combining facts across many pages, or queries that ask about something the PDF simply does not cover (like the taxes question). For those, we would need query rewriting, a larger `k` with a re-ranker, or a similarity-score threshold to abstain.

## 🏗️ Activity: Tune Retrieval Quality

Improve retrieval quality by changing one or more of these values:

- The chunk size
- The chunk overlap
- The retrieval `k`
- The wording of the retrieval query

Suggested workflow:

1. Pick one test question.
2. Inspect the retrieved chunks and scores.
3. Change one retrieval setting.
4. Rebuild the splitter and vector store.
5. Compare whether the retrieved chunks became more relevant.

When you are done, write down what changed and whether the final answer improved.

In [17]:
# Activity workspace
# Goal: see whether smaller chunks + a larger k improves retrieval for a
# specific, fact-shaped question.

test_question = "How often should senior cats be examined by a veterinarian?"

# --- Baseline: existing 1000/200 chunks, k=4 -------------------------------
print("=" * 80)
print("BASELINE: chunk_size=1000, chunk_overlap=200, k=4")
print("=" * 80)
_ = display_retrieval_results(test_question, k=4)

# --- Variant: smaller, more focused chunks, larger k -----------------------
tuned_chunk_size = 500
tuned_chunk_overlap = 100
tuned_k = 6

tuned_splitter = RecursiveCharacterTextSplitter(
    chunk_size=tuned_chunk_size,
    chunk_overlap=tuned_chunk_overlap,
    add_start_index=True,
)
tuned_splits = tuned_splitter.split_documents(pages)
print(f"\nRe-chunked into {len(tuned_splits)} chunks "
      f"(chunk_size={tuned_chunk_size}, overlap={tuned_chunk_overlap}).")

tuned_vector_store = QdrantVectorStore.from_documents(
    documents=tuned_splits,
    embedding=embeddings,
    location=":memory:",
    collection_name="cat_health_guidelines_tuned",
    force_recreate=True,
)

print("\n" + "=" * 80)
print(f"TUNED: chunk_size={tuned_chunk_size}, chunk_overlap={tuned_chunk_overlap}, k={tuned_k}")
print("=" * 80)
tuned_results = tuned_vector_store.similarity_search_with_score(test_question, k=tuned_k)
for index, (doc, score) in enumerate(tuned_results, start=1):
    page = doc.metadata.get("page")
    page_display = page + 1 if isinstance(page, int) else "unknown"
    preview = doc.page_content[:350].replace("\n", " ")
    print(f"Source {index} | score={score:.3f} | page={page_display}")
    print(preview)
    print("-" * 80)

# --- Compare final answers -------------------------------------------------
def answer_with_store(store, question, k):
    scored = store.similarity_search_with_score(question, k=k)
    context = format_context(scored)
    return rag_chain.invoke({"context": context, "question": question})

print("\nBASELINE ANSWER:\n", answer_with_store(vector_store, test_question, 4))
print("\nTUNED ANSWER:\n", answer_with_store(tuned_vector_store, test_question, tuned_k))

BASELINE: chunk_size=1000, chunk_overlap=200, k=4
Source 1 | score=0.706 | page=6 | start_index=0
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care recommenda- tions can be developed. Discussion Items for All Life Stages The Task Force reco
--------------------------------------------------------------------------------
Source 2 | score=0.608 | page=10 | start_index=1637
Young Adult Cats Y oung adult cats do not require as frequent routine medical care as kittens, so it is integral to educate the client about why regular healthcare examinations remain so important. Routine examinations can help identify behavioral changes or medical concerns that may affect a cat ’s health long before they become signi ﬁcant, painf
-------------------------------------------------------------------

### 🏗️ Activity Notes

- **Setting changed:** chunk size 1000 → 500, overlap 200 → 100, and `k` 4 → 6.
- **Before:** With 1,000-char chunks the top results for "How often should senior cats be examined?" included broad sections covering examination focus, behavior, and grooming. The exact frequency ("at least every 6 months") was present but embedded inside a large, multi-topic chunk, so the top-1 score was modest and the chunk carried a lot of unrelated text.
- **After:** With 500-char chunks the chunk that actually states the exam frequency for senior cats becomes more topically pure, so its embedding aligns better with the query and it tends to move up in rank with a higher similarity score. Bumping `k` to 6 makes up for the smaller chunks by giving the LLM enough surrounding context to assemble a complete answer.
- **Did retrieval improve? Why or why not?** Yes for a *fact-shaped* question like this one — smaller chunks reward queries that are specific because the embedding isn't averaged across many topics. The tradeoff is that for *broad* questions ("what is preventive care?") the smaller chunks are too fragmentary, so we'd want to bump `k` further or go back to larger chunks. The general lesson: chunk size should be tuned **to the shape of the questions you expect**, not picked once globally. A real system typically uses a re-ranker on top of vector search to get the best of both worlds.